<a href="https://colab.research.google.com/github/drhigh4444-code/Mock-teach/blob/main/LLM_Planning_Systems_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM Internals & Planning Systems — Live Lab
### MSAGI · Applied Agentic AI · Simplilearn

**What we build:** a single agent that takes a goal in plain English, **plans** the steps,
**calls a tool** when it needs one, observes the result, and **loops until done** — the
ReAct pattern (Reason + Act) from the slides.

We write the loop **in plain Python** so you can see the mechanism directly, rather than
hiding it inside a framework. The only dependency is the official model SDK.

> Run each cell top to bottom (Shift+Enter). The printed **Thought / Action / Observation**
> trace IS the lesson — you are watching the model plan.


## Step 1 — Install the SDK
One small dependency. ~10 seconds.

In [1]:
!pip install -q openai
print("Ready.")

Ready.


## Step 2 — Set your API key
Paste the key into the hidden prompt when it appears. It is **not** stored in the notebook,
and it will **not** appear on your shared screen.

> Teaching note: any function-calling model works. We keep this vendor-neutral — the
> *planning behavior* is the point, not the model brand.

In [3]:
import os

# Preferred: store your key once in Colab Secrets (left sidebar, key icon)
# as OPENAI_API_KEY with "Notebook access" ON. Then it loads automatically —
# nothing to type, nothing shown on screen.
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("Key loaded from Colab Secrets.")
except Exception:
    # Fallback: hidden prompt (key is not stored in the notebook or shown on screen)
    import getpass
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key (hidden): ")
    print("Key set for this session.")

from openai import OpenAI
client = OpenAI()
MODEL = "gpt-4o-mini"   # any tool-calling model is fine
print("Client ready.")

Key loaded from Colab Secrets.
Client ready.


## Step 3 — Define a tool: get_weather
A tool is just a Python function the agent is allowed to call. We give it a
**`get_weather`** tool — the same example from the lesson slides.

> **Note — this is a *mock* tool.** It returns a fixed forecast instead of calling a
> live weather service. In production it would hit a real weather API, but the agent's
> job is identical: decide *when* to call it and *what to do* with the result. That
> decision is the planning we want to see.

The **description** in the schema is not decoration — the model reads it to decide
*when* to use the tool. In function calling, the description is the interface.

In [4]:
def get_weather(city: str, date: str = "tomorrow") -> str:
    """Get the weather forecast for a city on a given date.
    (Mock tool: returns a fixed forecast for teaching purposes.)"""
    # A real implementation would call a weather API here.
    forecast = {"city": city, "date": date,
                "condition": "rain", "chance": "70%", "temp_c": 21}
    return (f"{forecast['city']} on {forecast['date']}: "
            f"{forecast['chance']} chance of {forecast['condition']}, "
            f"around {forecast['temp_c']}C.")

TOOLS = {"get_weather": get_weather}

# The schema the model sees. NOTE how much the description matters.
TOOL_SCHEMAS = [
    {"type": "function", "function": {
        "name": "get_weather",
        "description": "Get the weather forecast for a city on a given date. Use this whenever a question depends on the weather.",
        "parameters": {"type": "object",
            "properties": {
                "city": {"type": "string", "description": "e.g. 'Delhi'"},
                "date": {"type": "string", "description": "e.g. 'tomorrow'"}},
            "required": ["city"]}}},
]
print("Tools registered:", list(TOOLS))

Tools registered: ['get_weather']


## Step 4 — The ReAct loop (write it ourselves)
This is the whole engine, in ~15 lines. Watch what it does:
**ask the model -> if it wants a tool, run it and feed the result back -> repeat -> until it answers.**
That loop is *agentic reasoning*. Everything else in the course builds on it.

In [5]:
import json

def run_agent(task, max_steps=6, verbose=True):
    messages = [
        {"role": "system", "content": "You are a planning agent. Break the goal into steps and use tools when helpful."},
        {"role": "user", "content": task},
    ]
    if verbose: print(f"GOAL: {task}\n" + "-"*60)
    for step in range(max_steps):
        resp = client.chat.completions.create(
            model=MODEL, messages=messages, tools=TOOL_SCHEMAS, temperature=0)
        msg = resp.choices[0].message
        if msg.tool_calls:
            messages.append(msg.model_dump())          # record the model's plan
            for tc in msg.tool_calls:
                name = tc.function.name
                args = json.loads(tc.function.arguments)
                if verbose: print(f"Thought:     I should use `{name}`.")
                result = TOOLS[name](**args)            # ACT: run the tool
                if verbose: print(f"Action:      {name}({args})")
                if verbose: print(f"Observation: {result}\n")
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": str(result)})
        else:
            if verbose: print(f"Final Answer: {msg.content}")
            return msg.content
    return "(stopped: hit max steps)"

print("Agent defined.")

Agent defined.


## Step 5 — Run it: the weather agent from the slides
This is the canonical example from the lesson: a question that depends on the weather.
The agent can't answer it natively — it must (1) recognize it needs the forecast,
(2) call `get_weather`, (3) reason about the result, and (4) advise.
**Watch the trace** — each Thought is the model planning, each Action is the tool call,
each Observation feeds the final answer.

In [6]:
task = "Should I carry an umbrella in Delhi tomorrow?"

answer = run_agent(task)
print("\n=== RETURNED ===")
print(answer)

GOAL: Should I carry an umbrella in Delhi tomorrow?
------------------------------------------------------------
Thought:     I should use `get_weather`.
Action:      get_weather({'city': 'Delhi', 'date': 'tomorrow'})
Observation: Delhi on tomorrow: 70% chance of rain, around 21C.

Final Answer: Yes, you should carry an umbrella in Delhi tomorrow, as there is a 70% chance of rain.

=== RETURNED ===
Yes, you should carry an umbrella in Delhi tomorrow, as there is a 70% chance of rain.


## Step 6 — Guided practice (your turn)
Change the goal and re-run. Try another weather-dependent question (a different city,
or "what should I wear?"), or one that needs **no** tool at all (watch the agent correctly
decide *not* to call the tool — restraint is also planning).

**Discussion:** where would this break? An ambiguous goal? A tool that errors? A goal
that needs information our tool doesn’t have? That last one is the bridge to
*Retrieval-Augmented Generation (RAG)* — next session.

In [7]:
your_task = "TODO: write your own multi-step goal here"

# Uncomment to run:
# print(run_agent(your_task))

## Key takeaways
- A **plan** is the model decomposing a goal into ordered steps.
- **Function calling** lets the model act through typed tools — the *description* is the contract.
- The **ReAct loop** (Thought -> Action -> Observation -> repeat) is the simplest durable
  single-agent pattern. You just built it in ~15 lines.
- Product lens: the trace you watched is also your **observability surface** — what you log,
  evaluate, and debug in production (LangSmith & friends, later modules).

**Next session:** our agent can act through tools — but how do we make sure its answers are
grounded in real, trustworthy knowledge rather than what it happens to remember?
→ **Retrieval-Augmented Generation (RAG).**
